# 谱聚类：用图论的眼睛看数据
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：03_无监督学习/聚类 | 源文件：谱聚类.py | 核心功能：图拉普拉斯特征分解、affinity/gamma 调参、非凸聚类

## 概述

谱聚类的思路与 KMeans 截然不同：它不直接对数据点聚类，而是先构建一个**相似度图**，然后对图的拉普拉斯矩阵做特征分解，把数据映射到一个新空间，在新空间中用 KMeans 聚类。

这个新空间中，原本非凸的簇结构（如月牙形）会变得线性可分——这就是谱聚类能处理任意形状簇的秘密。

脚本在月牙形、同心圆和球形数据上对比了谱聚类和 KMeans，展示了谱聚类在非凸数据上的压倒性优势。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.cluster import SpectralClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

## 数学原理

### 1. 图拉普拉斯矩阵

**代码对应**：`SpectralClustering(n_clusters=k, affinity="rbf")` 内部构建相似度图并计算拉普拉斯矩阵。

构建相似度矩阵 $\mathbf{W}$（$W_{ij}$ 为样本 $i$ 和 $j$ 的相似度），度矩阵 $\mathbf{D} = \text{diag}(d_1, \ldots, d_n)$，其中 $d_i = \sum_j W_{ij}$。

**图拉普拉斯矩阵**：

$$\mathbf{L} = \mathbf{D} - \mathbf{W}$$

**归一化拉普拉斯**（Normalized Laplacian）：

$$\mathbf{L}_{\text{sym}} = \mathbf{D}^{-1/2}\mathbf{L}\mathbf{D}^{-1/2} = \mathbf{I} - \mathbf{D}^{-1/2}\mathbf{W}\mathbf{D}^{-1/2}$$

### 2. 谱聚类的核心思想

谱聚类将聚类问题转化为**图切割**问题。最优切割（Normalized Cut）：

$$\text{NCut}(C_1, \ldots, C_K) = \sum_{k=1}^{K}\frac{\text{cut}(C_k, \bar{C}_k)}{\text{vol}(C_k)}$$

其中 $\text{cut}(A, B) = \sum_{i \in A, j \in B}W_{ij}$，$\text{vol}(A) = \sum_{i \in A}d_i$。

NCut 是 NP-hard 问题。谱聚类通过**松弛化**将其变为特征值问题：

$$\mathbf{L}_{\text{sym}}\mathbf{u} = \lambda\mathbf{u}$$

取最小的 $K$ 个非零特征值对应的特征向量，构成 $n \times K$ 的嵌入矩阵，然后对嵌入后的行向量做 KMeans 聚类。

### 3. 算法流程

1. 构建相似度矩阵 $\mathbf{W}$（RBF 核：$W_{ij} = \exp(-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2)$）
2. 计算归一化拉普拉斯 $\mathbf{L}_{\text{sym}}$
3. 求 $\mathbf{L}_{\text{sym}}$ 的最小 $K$ 个特征向量，组成矩阵 $\mathbf{U} \in \mathbb{R}^{n \times K}$
4. 对 $\mathbf{U}$ 的行向量做 KMeans 聚类

### 4. 为什么谱聚类能处理非凸簇？

KMeans 在原始空间中用欧氏距离，只能发现球形簇。谱聚类通过拉普拉斯特征向量将数据映射到一个新空间（谱嵌入空间），在该空间中非凸簇变为线性可分。

**直觉**：相似度图上的"瓶颈"（低切割边）对应数据的自然分割。拉普拉斯矩阵的特征向量恰好捕捉了这些瓶颈。

### 5. 时间复杂度

- 构建相似度矩阵：$O(n^2 p)$
- 特征值分解：$O(n^3)$（精确）或 $O(nK^2)$（稀疏矩阵迭代法）
- KMeans：$O(nK^2 T)$

总复杂度通常为 $O(n^3)$，不适合大数据集。

### 1. 生成不同形状的数据

运行 1. 生成不同形状的数据 的代码，观察算法在该环节的行为。

In [ ]:
X_moon, y_moon = make_moons(n_samples=300, noise=0.1, random_state=42)
X_circle, y_circle = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
X_blob, y_blob = make_blobs(n_samples=300, centers=3, random_state=42)

### 2. 基础谱聚类

运行 2. 基础谱聚类 的代码，观察算法在该环节的行为。

In [ ]:
X = StandardScaler().fit_transform(X_moon)
sc = SpectralClustering(n_clusters=2, affinity="rbf", random_state=42)
labels = sc.fit_predict(X)

print("=== 谱聚类（月亮形数据）===")
print(f"簇标签分布: {np.bincount(labels)}")
print(f"Silhouette: {silhouette_score(X, labels):.4f}")
print(f"ARI: {adjusted_rand_score(y_moon, labels):.4f}")

### 3. affinity（相似度度量）

运行 3. affinity（相似度度量） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== affinity 对比 ===")
X_c = StandardScaler().fit_transform(X_circle)
for aff in ["rbf", "nearest_neighbors"]:
    if aff == "nearest_neighbors":
        sc_a = SpectralClustering(n_clusters=2, affinity=aff, n_neighbors=10, random_state=42)
# --- 继续 ---
    else:
        sc_a = SpectralClustering(n_clusters=2, affinity=aff, random_state=42)
    labels_a = sc_a.fit_predict(X_c)
    sil = silhouette_score(X_c, labels_a)
    ari = adjusted_rand_score(y_circle, labels_a)
# --- 输出结果 ---
    print(f"  affinity={aff:>17}: Silhouette={sil:.4f}, ARI={ari:.4f}")

### 4. gamma 参数（RBF 核）

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== gamma 参数对比（RBF 核）===")
for gamma in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    sc_g = SpectralClustering(n_clusters=2, affinity="rbf", gamma=gamma, random_state=42)
    labels_g = sc_g.fit_predict(X)
    sil = silhouette_score(X, labels_g)
# --- 继续 ---
    ari = adjusted_rand_score(y_moon, labels_g)
    print(f"  gamma={gamma:>5}: Silhouette={sil:.4f}, ARI={ari:.4f}")

### 5. n_neighbors（nearest_neighbors affinity）

运行 5. n_neighbors（nearest_neighbors affinity） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== n_neighbors 参数对比 ===")
for nn in [5, 10, 15, 20, 30]:
    sc_n = SpectralClustering(n_clusters=2, affinity="nearest_neighbors",
                               n_neighbors=nn, random_state=42)
    labels_n = sc_n.fit_predict(X)
# --- 继续 ---
    sil = silhouette_score(X, labels_n)
    print(f"  n_neighbors={nn:>2}: Silhouette={sil:.4f}")

### 6. 不同数据集对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 不同数据集上谱聚类 vs KMeans ===")
from sklearn.cluster import KMeans
datasets = {"月亮形": (X_moon, y_moon, 2), "同心圆": (X_circle, y_circle, 2), "球形": (X_blob, y_blob, 3)}
for name, (X_d, y_d, k) in datasets.items():
    X_d = StandardScaler().fit_transform(X_d)
# --- 继续 ---
    sc_d = SpectralClustering(n_clusters=k, affinity="rbf", random_state=42)
    km_d = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_sc = sc_d.fit_predict(X_d)
    labels_km = km_d.fit_predict(X_d)
    ari_sc = adjusted_rand_score(y_d, labels_sc)
# --- 继续 ---
    ari_km = adjusted_rand_score(y_d, labels_km)
    print(f"  {name}: 谱聚类 ARI={ari_sc:.4f}, KMeans ARI={ari_km:.4f}")

print("\n=== 谱聚类要点 ===")
print("1. 构建相似度图（affinity matrix）")
print("2. 计算拉普拉斯矩阵的前 K 个特征向量")
print("3. 在特征向量空间中执行 KMeans 聚类")
print("- 优点：能发现非凸形状的簇（不像 KMeans 只能发现球形簇）")
# --- 输出结果 ---
print("- 缺点：计算量大 O(n³)、对参数（gamma/n_neighbors）敏感")
print("- affinity='rbf' 的 gamma 控制相似度衰减速度")
print("- affinity='nearest_neighbors' 用 K 近邻图构建相似度矩阵")

## 关键代码解释

### affinity 参数

```python
for aff in ["rbf", "nearest_neighbors"]:
    sc_a = SpectralClustering(n_clusters=2, affinity=aff)
```

- **rbf**：高斯核相似度，gamma 控制衰减速度
- **nearest_neighbors**：K 近邻图，n_neighbors 控制邻域大小

### 与 KMeans 的对比

```python
sc_d = SpectralClustering(n_clusters=k, affinity="rbf")
km_d = KMeans(n_clusters=k)
# 在月牙形/同心圆上，谱聚类 >> KMeans
# 在球形数据上，两者相当
```

## 使用示例

```python
from sklearn.cluster import SpectralClustering
sc = SpectralClustering(n_clusters=2, affinity="rbf", gamma=1.0, random_state=42)
labels = sc.fit_predict(X)
```

## 注意事项

1. **计算量大 O(n³)**：不适合大数据集
2. **对 gamma/n_neighbors 敏感**
3. **必须特征缩放**
4. **在球形数据上不比 KMeans 好**：没必要用大炮打蚊子

## 总结与延伸

以上代码展示了 **谱聚类** 的完整流程。

**解读要点**：
- 关注 **轮廓系数（Silhouette Score）**，越接近 1 越好
- 观察聚类中心是否与数据分布吻合
- 对比不同 K 值的聚类效果

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **归一化切（Normalized Cut）**：谱聚类的经典目标函数
- **Nyström 近似**：用采样近似大规模相似度矩阵，加速谱聚类
- **深度谱聚类**：用自编码器学习相似度矩阵
- **图神经网络**：谱聚类的"深度学习版本"
